<a href="https://colab.research.google.com/github/talhanoor23/my_first_repository/blob/main/News_Ag_Bot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# !pip install langgraph langchain langsmith python-dotenv langchain-groq langchain_tavily langchain-community
# !pip install --upgrade google-generativeai langchain-google-genai langchain

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm = ChatGoogleGenerativeAI(model="models/gemini-2.5-flash")

In [ ]:
# # Install dependencies
# !pip install transformers accelerate

# import torch
# from transformers import AutoTokenizer, AutoModelForCausalLM

# # Model name on HuggingFace
# model_name = "cerebras/Cerebras-GPT-1.3B"

# # Load tokenizer & model (automatic download in Colab)
# tokenizer = AutoTokenizer.from_pretrained(model_name)
# llm = AutoModelForCausalLM.from_pretrained(
#     model_name,
#     device_map="auto",       # automatic GPU support
#     torch_dtype=torch.float16 # faster and lower memory
# )

In [ ]:
from typing import Annotated, Dict, Any
from typing_extensions import TypedDict
from langgraph.graph.message import add_messages

class State(TypedDict):
    messages: Annotated[list, add_messages]
    # reasoning: Dict[str, Any] | None
    # news_done: bool

In [ ]:
os.environ["NEWS_API_KEY"]="70e32b43f4124e2895941e1bb7a109a2"
apiKey=os.environ["NEWS_API_KEY"]

In [ ]:
# import requests
# from typing import Optional, List, Dict
# from datetime import datetime, timedelta, timezone




# from langchain.tools import tool
# # from langchain.chat_models import ChatOpenAI


# NEWS_API_URL = "https://newsapi.org/v2/everything"


# def fetch_financial_news(minutes: int, topic_hint: Optional[str]) -> List[Dict]:
#     from_time = (datetime.now(timezone.utc) - timedelta(minutes=minutes)).isoformat()

#     params = {
#         # "category": "business",
#         "language": "en",
#         "pageSize": 100,
#         "from": from_time,
#         "apiKey": os.getenv("NEWS_API_KEY"),
#     }

#     if topic_hint:
#         params["q"] = topic_hint

#     response = requests.get(NEWS_API_URL, params=params, timeout=10)
#     response.raise_for_status()

#     return response.json().get("articles", [])


In [ ]:
# def explain_importance(llm, title: str, description: str) -> str:
#     prompt = f"""
# YouAre a financial analyst.
# Explain briefly (1–2 lines) why this news might matter to markets.

# Title: {title}
# Description: {description}
# """
#     return llm.invoke(prompt).content.strip()

In [ ]:
# tool
# def financial_news_research(
#     minutes: int = 60,
#     topic_hint: Optional[str] = None,
#     depth: str = "normal"
# ) -> List[Dict]:

#     """
#     Fetches recent financial news and explains why it matters.

#     Args:
#         minutes: How recent the news should be (e.g., last 60 minutes)
#         topic_hint: Optional topic hint like 'rates', 'banks', 'inflation'
#         depth: 'quick' or 'normal'

#     Returns:
#         List of important financial news items
#     """

#     print("🚨 TOOL CALLED!")
#     print(f"Parameters: minutes={minutes}, topic_hint={topic_hint}, depth={depth}")

#     articles = fetch_financial_news(minutes, topic_hint)

#     if not articles:
#         print("⚠️ No articles found. Skipping LLM calls.")
#         return []

#     results = []

#     for article in articles:
#         print("🤖 Calling LLM...")
#         print(f"Title: {article.get('title')}")
#         print(f"Description: {article.get('description', '')}")
#         title = article.get("title")
#         description = article.get("description") or ""

#         if not title:
#             continue

#         # OPTIONAL: move this OUT of the tool later
#         importance = explain_importance(llm, title, description)

#         results.append({
#             "headline": title,
#             "why_important": importance,
#             "source": article.get("source", {}).get("name"),
#             "published_at": article.get("publishedAt"),
#             "url": article.get("url")
#         })

#     return results

In [ ]:
# import os
# import requests
# from typing import Optional, List, Dict
# from datetime import datetime, timedelta, timezone
# from langchain.tools import tool

# NEWS_API_URL = "https://newsapi.org/v2/everything"


# # ----------------------------
# # Fetch news (NO LLM here)
# # ----------------------------
# def fetch_financial_news(minutes: int, topic_hint: Optional[str]) -> List[Dict]:
#     from_time = (datetime.now(timezone.utc) - timedelta(days=10)).isoformat()

#     params = {
#         "language": "en",
#         "pageSize": 10,
#         "from": from_time,
#         "apiKey": os.getenv("NEWS_API_KEY"),
#     }

#     if topic_hint:
#         params["q"] = topic_hint

#     response = requests.get(NEWS_API_URL, params=params, timeout=10)
#     response.raise_for_status()
#     return response.json().get("articles", [])


# # ----------------------------
# # Tool (LLM called once)
# # ----------------------------
# tool
# def financial_news_research(
#     minutes: int = 1440,
#     topic_hint: Optional[str] = None
# ) -> List[Dict]:
#     """
#     Fetch financial news and explain why it matters.
#     LLM is called ONCE.
#     """

#     print("🚨 TOOL CALLED")
#     print(f"minutes={minutes}, topic_hint={topic_hint}")

#     # ✅ CRITICAL FIX
#     if not topic_hint or len(topic_hint.strip()) < 3:
#         topic_hint = "finance OR stock OR market OR economy"


#     articles = fetch_financial_news(minutes, topic_hint)

#     if not articles:
#         return []

#     prompt_blocks = []
#     valid_articles = []

#     for i, a in enumerate(articles, start=1):
#         title = a.get("title")
#         description = a.get("description") or ""

#         if not title:
#             continue

#         valid_articles.append(a)
#         prompt_blocks.append(
#             f"""Article {i}
# Title: {title}
# Description: {description}
# """
#         )

#     prompt = f"""
# You are a financial market analyst.

# For EACH article below, explain in one sentence
# why it matters to financial markets.

# Return a numbered list matching the article order.

# {chr(10).join(prompt_blocks)}
# """

#     print("🤖 Calling LLM ONCE")
#     response = llm.invoke(prompt).content.strip()

#     explanations = [
#         line.split(".", 1)[-1].strip()
#         for line in response.splitlines()
#         if line.strip()
#     ]

#     results = []
#     for i, article in enumerate(valid_articles):
#         results.append({
#             "headline": article.get("title"),
#             "why_important": explanations[i] if i < len(explanations) else "No explanation",
#             "source": article.get("source", {}).get("name"),
#             "published_at": article.get("publishedAt"),
#             "url": article.get("url"),
#         })

#     return results


In [ ]:
!pip install keybert
from keybert import KeyBERT

kw_model = KeyBERT(model="all-MiniLM-L6-v2")

def extract_keywords(text: str, top_k: int = 5) -> str:
    keywords = kw_model.extract_keywords(
        text,
        keyphrase_ngram_range=(1, 2),
        stop_words="english",
        top_n=top_k
    )
    return " OR ".join([kw[0] for kw in keywords])

In [ ]:
import os
import requests
from typing import Optional, List, Dict
from datetime import datetime, timedelta, timezone
from langchain.tools import tool

NEWS_API_URL = "https://newsapi.org/v2/everything"

In [ ]:
def fetch_financial_news(minutes: int, query: str) -> List[Dict]:
    from_time = (datetime.now(timezone.utc) - timedelta(days=7)).isoformat()

    params = {
        "language": "en",
        "pageSize": 10,
        "from": from_time,
        "q": query,
        "apiKey": os.getenv("NEWS_API_KEY"),
    }

    response = requests.get(NEWS_API_URL, params=params, timeout=10)
    response.raise_for_status()
    return response.json().get("articles", [])


In [ ]:
@tool
def financial_news_research(
    minutes: int = 1440,
    user_query: str = "",
    state: Optional[dict] = None
) -> dict:
    """
    Fetch financial news and mark state when done.
    """
    print("🚨 TOOL CALLED")
    if not user_query.strip():
        user_query = "finance market economy stocks"

    keyword_query = extract_keywords(user_query)
    articles = fetch_financial_news(minutes, keyword_query)

    if not articles:
        return {
            "results": [],
            "status": "NO_DATA",
            "message": "No relevant news found"
        }

    # LLM call happens once
    prompt_blocks = []
    valid_articles = []
    for i, a in enumerate(articles, start=1):
        title = a.get("title")
        if not title:
            continue
        valid_articles.append(a)
        description = a.get("description") or ""
        prompt_blocks.append(f"Article {i}\nTitle: {title}\nDescription: {description}\n")

    prompt = f"""
You are a financial market analyst.

For EACH article below, explain in one sentence
why it matters to financial markets.

Return a numbered list matching the article order.

{chr(10).join(prompt_blocks)}
"""
    response = llm.invoke(prompt).content.strip()
    explanations = [
        line.split(".", 1)[-1].strip()
        for line in response.splitlines()
        if line.strip()
    ]

    results = [
        {
            "headline": a.get("title"),
            "why_important": explanations[i] if i < len(explanations) else "No explanation",
            "source": a.get("source", {}).get("name"),
            "published_at": a.get("publishedAt"),
            "url": a.get("url"),
        }
        for i, a in enumerate(valid_articles)
    ]
    return results

    # # Return everything in one dictionary, including the state flag
    # return {
    #     "results": results,
    #     "news_done": True,
    #     "state": state
    # }


In [ ]:
# SUPERVISOR_PROMPT = """
# You are a financial AI assistant.

# Capabilities:
# - Fetch real-time financial news
# - Explain trading and investing concepts
# - Provide educational guidance (not financial advice)

# Rules:
# - Use tools when real-time, factual, or external data is required
# - Answer directly for conceptual or educational questions
# - NEVER invent news or events
# - Choose the most appropriate tool based on user intent
# """

In [ ]:
# REASONING_PROMPT = """
# You are a financial reasoning analyst.

# You are given recent financial news summaries.
# Your task is to reason about their implications.

# User question:
# {user_question}

# News summaries:
# {news_summaries}

# Instructions:
# - Identify cause–effect relationships
# - Separate short-term vs long-term impact
# - Explain who is affected (markets, sectors, people)
# - Mention what to watch next

# Rules:
# - Be analytical, not descriptive
# - Do not repeat headlines
# - Do not hallucinate facts
# - Use clear bullet points
# """

In [ ]:
# def reasoning_agent(llm, news_items, user_question):
#     """
#     Reasons about the implications of financial news.
#     """
#     print("🤖 REASONING AGENT CALLED")
#     if not news_items:
#         return "No data available to reason about."

#     summaries = ""
#     for i, n in enumerate(news_items, start=1):
#         summaries += f"""
# {i}. Headline: {n['headline']}
#    Context: {n['why_important']}
# """

#     prompt = REASONING_PROMPT.format(
#         user_question=user_question,
#         news_summaries=summaries
#     )

#     return llm.invoke(prompt).content.strip()

In [ ]:
# from langchain_core.tools import tool

# tool
# def financial_reasoning(
#     user_question: str,
#     context: str = ""
# ) -> str:
#     """
#     Analyze financial implications using provided context.
#     No external data fetching.
#     """

#     prompt = f"""
# You are a financial reasoning agent.

# Rules:
# - Do NOT fetch news
# - Do NOT invent facts
# - Use ONLY provided context
# - Explain cause–effect clearly
# - Separate short-term and long-term impacts

# Context:
# {context if context else "No external context provided."}

# Question:
# {user_question}

# Provide structured reasoning.
# """
#     print("🤖 REASONING AGENT CALLED")
#     return llm.invoke(prompt).content.strip()

In [ ]:
REASONING_SCHEMA = {
    "summary": str,
    "sentiment": "bullish | bearish | neutral",
    "confidence": float,  # 0.0 - 1.0
    "key_points": list[str],
    "risks": list[str],
    "time_horizon": "short_term | mid_term | long_term"
}

In [ ]:
REASONING_PROMPT = """
You are a financial reasoning agent.

Analyze the user's query and available context.

Return STRICT JSON matching this schema:
{
  "summary": string,
  "sentiment": "bullish" | "bearish" | "neutral",
  "confidence": number (0 to 1),
  "key_points": [string],
  "risks": [string],
  "time_horizon": "short_term" | "mid_term" | "long_term"
}

Rules:
- NO markdown
- NO explanations
- NO extra text
- If uncertain, lower confidence
"""


In [ ]:
import json
from langchain.tools import tool
from langchain_core.messages import HumanMessage

@tool
def structured_reasoning_tool(user_query: str) -> dict:
    """
    Perform structured financial reasoning on a user query.

    Input:
    - user_query: The user's financial or market-related question.

    Output (JSON):
    {
        "summary": string,
        "sentiment": "bullish" | "bearish" | "neutral",
        "confidence": number (0 to 1),
        "key_points": [string],
        "risks": [string],
        "time_horizon": "short_term" | "mid_term" | "long_term"
    }
    """
    print("🤖 REASONING AGENT CALLED")
    combined_prompt = f"{REASONING_PROMPT}\n\nUser Query: {user_query}"
    response = llm.invoke([
        HumanMessage(content=combined_prompt)
    ])
    print(response.content)

    try:
        return json.loads(response.content)
    except Exception:
        return {
            "summary": "Unable to generate structured reasoning",
            "sentiment": "neutral",
            "confidence": 0.0,
            "key_points": [],
            "risks": ["Parsing failure"],
            "time_horizon": "short_term"
        }

In [ ]:
# SUPERVISOR_PROMPT = """
# You are a financial AI assistant.

# Capabilities:
# - Fetch real-time financial news
# - Explain trading and investing concepts
# - Provide educational guidance (not financial advice)

# Rules:
# - Use tools ONLY when external or real-time data is required
# - Each tool may be called at most ONCE per user query
# - After a tool returns any result, you MUST consider the information sufficient
# - You are NOT allowed to call the same tool again for the same query
# - If information is incomplete, answer with uncertainty instead of retrying
# - NEVER invent news or events
# - Choose the most appropriate tool based on user intent
# """

In [ ]:
SUPERVISOR_PROMPT = """
You are a financial AI system composed of specialized agents.

Your goal is to provide accurate, well-reasoned, and up-to-date responses.

You have access to:
- A Financial News Agent (real-time information)
- A Reasoning Agent (macro, cause-effect, implications, synthesis)
- Educational knowledge (no tools required)
If reasoning is needed, donot reason it yourself, first call the news the you must have to call the Reasoning Agent

General Principles:
- Do NOT invent facts, news, or events
- If real-time or external data is required, fetch it before reasoning
- If reasoning is required, base it strictly on known facts or fetched data
- Prefer multi-step reasoning over shallow answers
- You may use more than one agent if the user query requires it
- Agents may consume outputs from other agents

Tool & Agent Usage Rules:
- Use tools ONLY when real-time or external information is needed or is required
- Each tool may be called at most ONCE per query
- After a tool returns data, treat it as sufficient
- Do NOT retry or re-call same tools again for the same query
- If information remains incomplete, state uncertainty clearly

Decision Logic:
1. Determine if the query requires real-time data
   - If yes → call Financial News Agent
2. Determine if the query requires implications, interpretation, or synthesis
   - If yes → call Financial News Agent for News then call Reasoning Agent (using available data)
3. If neither is required → answer directly

Output:
- Provide a coherent final answer to the user
- Do not expose internal routing or agent decisions
- Do not mention tools unless explicitly asked
"""

In [ ]:
# SUPERVISOR_PROMPT = """
# You are a financial AI supervisor.

# Your job:
# - Decide whether the user needs:
#   1) financial news
#   2) financial reasoning
#   3) both (news first, then reasoning)
#   4) or a direct answer

# Rules:
# - Use tools ONLY if required
# - Each tool may be called at most once
# - Never invent news
# - If unsure, answer directly with uncertainty
# """

In [ ]:
tools=[
        financial_news_research,
        structured_reasoning_tool
    ]
llm_with_tools = llm.bind_tools(tools)

In [ ]:
# from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# # Example: WizardLM 7B (instruction-tuned)
# # model_name = "TheBloke/WizardLM-7B-uncensored-GGUF" # Original problematic line

# # Fix: Use a standard Hugging Face model name for AutoTokenizer to find the tokenizer files.
# # For GGUF models, tokenizer files are often in the original model's repository or require specific handling.
# # Using 'gpt2' as a widely supported placeholder for a functional model.
# model_name = "gpt2"

# tokenizer = AutoTokenizer.from_pretrained(model_name)
# model = AutoModelForCausalLM.from_pretrained(model_name, device_map="auto")

# # Create a text generation pipeline
# free_llm = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=100)


In [ ]:
# INTENT_TOOL_MAP = {
#     "NEWS_ONLY": ["financial_news_research"],
#     "REASONING_ONLY": ["financial_reasoning"],
#     "NEWS_AND_REASONING": ["financial_news_research", "financial_reasoning"]
#     # Supervisor answers directly without calling tools
# }


In [ ]:
# def intent_classifier(query: str) -> str:
#     """
#     Free model detects intent from the query.
#     Returns one of: NEWS_ONLY, REASONING_ONLY, NEWS_AND_REASONING, GENERAL_CHAT
#     """
#     free_model_prompt = f"""
# Classify the following user query into one of the labels:
# - NEWS_ONLY
# - REASONING_ONLY
# - NEWS_AND_REASONING
# - GENERAL_CHAT

# Query: "{query}"

# Return ONLY the label, no explanations.
# """
#     # TextGenerationPipeline objects are called directly with the prompt string
#     response = free_llm(free_model_prompt)

#     # Extract the generated text. It will contain the prompt itself.
#     generated_text = response[0]['generated_text']

#     # The prompt instructs to "Return ONLY the label".
#     # We need to remove the original prompt from the generated text to get just the label.
#     if generated_text.startswith(free_model_prompt):
#         raw_label = generated_text[len(free_model_prompt):].strip()
#     else:
#         # Fallback if model doesn't exactly echo the prompt or misbehaves
#         raw_label = generated_text.strip()

#     valid_intents = ["NEWS_ONLY", "REASONING_ONLY", "NEWS_AND_REASONING", "GENERAL_CHAT"]
#     for intent in valid_intents:
#         if intent in raw_label: # Use 'in' for flexibility
#             return intent

#     # Default to GENERAL_CHAT if no specific intent is clearly identified
#     return "GENERAL_CHAT"

In [ ]:
# def intent_classifier(query: str) -> str:
#     """
#     Free model detects intent from the query.
#     Returns one of: NEWS_ONLY, REASONING_ONLY, NEWS_AND_REASONING, GENERAL_CHAT
#     """
#     free_model_prompt = f"""
# Classify the following user query into one of the labels:
# - NEWS_ONLY
# - REASONING_ONLY
# - NEWS_AND_REASONING


# Query: "{query}"

# Return ONLY the label, no explanations.
# """
#     response = free_llm(free_model_prompt)
#     generated_text = response[0]['generated_text']

#     # Take first word/token and match to valid intents
#     raw_label = generated_text.strip().split()[0]
#     valid_intents = ["NEWS_ONLY", "REASONING_ONLY", "NEWS_AND_REASONING"]

#     if raw_label in valid_intents:
#         return raw_label
#     return "GENERAL_CHAT"


In [ ]:
# from langchain_core.messages import SystemMessage

# def supervisor_node(state: State):
#     print("🧠 SUPERVISOR CALLED")
#     messages = state["messages"]

#     if not messages:
#         messages = [SystemMessage(content=SUPERVISOR_PROMPT)]

#     user_query = messages[-1].content

#     # 1️⃣ Detect intent using free model
#     intent = intent_classifier(user_query)
#     print(f"🎯 DETECTED INTENT: {intent}")

#     # 2️⃣ Determine tools to call based on intent
#     tools_to_call = INTENT_TOOL_MAP.get(intent, [])
#     print(f"🛠 TOOLS TO CALL: {tools_to_call}")

#     # 3️⃣ Prepare supervisor prompt for tool-bound LLM
#     supervisor_prompt = f"""
# You are a financial AI supervisor.

# User Query: {user_query}
# Detected Intent: {intent}
# Tools to call: {', '.join(tools_to_call) if tools_to_call else 'None'}

# Rules:
# - Follow the detected intent
# - Call each tool at most once
# - If no tools are needed, answer directly
# - Use all conversation history
# """
#     # 4️⃣ Call tool-bound LLM
#     response = llm_with_tools.invoke([SystemMessage(content=supervisor_prompt)])

#     return {
#         "messages": messages + [response],
#         "intent": intent,
#         "tools_called": tools_to_call
#     }


In [ ]:
# from langchain_core.messages import SystemMessage, HumanMessage

# def supervisor_node(state: State):
#     print("🧠 SUPERVISOR CALLED")
#     messages = state.get("messages", [])

#     if not messages:
#         messages = [SystemMessage(content=SUPERVISOR_PROMPT)]

#     # 1️⃣ Get latest user query
#     user_query = messages[-1].content if messages else ""

#     # 2️⃣ Detect intent using free model
#     intent = intent_classifier(user_query)
#     print(f"🎯 DETECTED INTENT: {intent}")

#     # 3️⃣ Determine tools to call based on intent
#     tools_to_call = INTENT_TOOL_MAP.get(intent, [])
#     print(f"🛠 TOOLS TO CALL: {tools_to_call}")

#     # 4️⃣ If no tools required, use a normal free LLM
#     if not tools_to_call:
#         prompt_direct = f"""
# You are a financial AI assistant.

# User Query: {user_query}

# Rules:
# - Answer based on your knowledge
# - Do NOT call any external tools
# - Be concise and clear
# """
#         # Use free LLM (not tool-bound) to avoid 'contents are required' error
#         response_text = free_llm(prompt_direct)[0]["generated_text"]
#         return {
#             "messages": messages + [HumanMessage(content=response_text)],
#             "intent": intent,
#             "tools_called": tools_to_call
#         }

#     # 5️⃣ Otherwise prepare prompt for tool-bound LLM
#     supervisor_prompt = f"""
# You are a financial AI supervisor.

# User Query: {user_query}
# Detected Intent: {intent}
# Tools to call: {', '.join(tools_to_call)}

# Rules:
# - Follow the detected intent
# - Call each tool at most once
# - Use all conversation history
# """
#     response = llm_with_tools.invoke([SystemMessage(content=supervisor_prompt)])

#     return {
#         "messages": messages + [response],
#         "intent": intent,
#         "tools_called": tools_to_call
#     }


In [ ]:
from langchain_core.messages import SystemMessage, AIMessage

def supervisor_node(state: State):
    print("🧠 SUPERVISOR CALLED")

    messages = state["messages"]
    if not messages:
        messages = [SystemMessage(content=SUPERVISOR_PROMPT)]

    # news = state.get("news")
    # if news["status"] == "NO_DATA":
    # # HARD STOP — no retry
    #     return {
    #     "final_answer": (
    #         "There is currently no reliable or confirmed news available "
    #         "on this topic. Based on historical patterns and general "
    #         "market behavior, here is a cautious, non-real-time perspective..."
    #     )
    # }


    # # 1️⃣ STOP CONDITION (THIS IS THE KEY)
    # if state.get("news_done"):
    #     print("🛑 News already fetched. Ending flow.")
    #     return {
    #         "messages": messages
    #     }

    # 2️⃣ Ensure system prompt
    if not messages or not isinstance(messages[0], SystemMessage):
        messages = [SystemMessage(content=SUPERVISOR_PROMPT)] + messages

    # 3️⃣ Call LLM (routing / tool decision)
    response = llm_with_tools.invoke(messages)

    # 5️⃣ Otherwise allow tool execution
    return {
        "messages": messages + [response]
    }

In [ ]:
from langgraph.graph import StateGraph,START,END
from langgraph.prebuilt import ToolNode
from langgraph.prebuilt import tools_condition


graph = StateGraph(State)

graph.add_node("respond", supervisor_node)
graph.add_node("tools",ToolNode(tools))

graph.set_entry_point("respond")

graph.add_conditional_edges(
    "respond",
    tools_condition
)
graph.add_edge("tools","respond")
graph.add_edge("respond", END)

app = graph.compile()

from IPython.display import Image, display
display(Image(app.get_graph().draw_mermaid_png()))

In [ ]:
# state = {"messages": [{"role": "user", "content": "Do we have to invest in tesla or not based on current conditions?"}]}
# result = app.invoke(state)
# print("Response:", result["messages"][-1].content)

In [ ]:
# from langchain_core.messages import HumanMessage

# result = app.invoke({
#     "messages": [
#         HumanMessage(content="How is bitcoin behaving these days? what to expect from bitcoin in future?")
#     ]
# })
# print("Response:", result["messages"][-1].content)

In [ ]:
# for msg in result["messages"]:
#     print(type(msg).__name__, msg)

In [ ]:
# from langchain_core.messages import HumanMessage

# result = app.invoke({
#     "messages": [
#         HumanMessage(content="What happens to markets during quantitative tightening?")
#     ]
# })
# print("Response:", result["messages"][-1].content)

In [ ]:
# from langchain_core.messages import HumanMessage

# result = app.invoke({
#     "messages": [
#         HumanMessage(content="Please provide me recent news on US100.")
#     ]
# })
# print("Response:", result["messages"][-1].content)

In [ ]:
# from langchain_core.messages import HumanMessage

# result = app.invoke({
#     "messages": [
#         HumanMessage(content="hello")
#     ]
# })
# print("Response:", result["messages"][-1].content)

In [ ]:
articles = fetch_financial_news(1440, "bitcoin")  # 60 minutes is ignored for now
print(f"Found {len(articles)} articles")
for a in articles:
    print("Title : ", a["title"], "-", a["source"]["name"])
    print("Description : ", a["description"])

In [ ]:
# articles_2 = fetch_financial_news(10000, "")  # 60 minutes is ignored for now
# print(f"Found {len(articles)} articles")
# for a in articles:
#     print("Title : ", a["title"], "-", a["source"]["name"])
#     print("Description : ", a["description"])

In [ ]:
keyword_query = extract_keywords("Please provide me recent news on US100.")
print(f"🔑 Extracted Keywords: {keyword_query}")

In [ ]:
articles = fetch_financial_news(10000, f"{keyword_query}")  # 60 minutes is ignored for now
print(f"Found {len(articles)} articles")
for a in articles:
    print("Title : ", a["title"], "-", a["source"]["name"])
    print("Description : ", a["description"])

In [ ]:
processed_news_items = financial_news_research.invoke(input={"user_query": "How is bitcoin behaving nowadays?"})

analysis = reasoning_agent(
    llm=llm,
    news_items=processed_news_items,
    user_question="What does this mean for markets?"
)

print(analysis)

In [ ]:
# financial_news_research.invoke(
#         {"minutes": 60, "topic_hint": "bitcoin", "depth": "normal"}
#     )